# Chicago 311 Service Requests - MongoDB Analiza

**Autor:** Marko Zelić IN48-2022  
**Predmet:** Sistemi baza podataka, FTN Novi Sad  
**Dataset:** Chicago 311 Service Requests (~1.3 GB, 12 CSV fajlova, 4M+ redova)

---

## 1. Imports

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import json
from pprint import pprint

import connection
import load_data
import base_queries
import optimized_queries
import optimize_schema
import query_executor

## 2. MongoDB konekcija

In [ ]:
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DATABASE_NAME = "chicago_311_db"

client, database = connection.connect_to_mongodb(MONGO_URI, MONGO_DATABASE_NAME)

## 3. Učitavanje podataka

Učitavanje 12 CSV fajlova u 6 normalizovanih kolekcija:
- `requests` - zajednička polja svih zahteva
- `locations` - deduplikovani geografski podaci
- `vehicle_details` - detalji napuštenih vozila
- `sanitation_details` - sanitarni detalji (glodari, đubre, kodne povrede)
- `environment_details` - infrastruktura i okruženje (rupe, svetla, grafiti, drveće)
- `building_details` - detalji napuštenih zgrada

In [ ]:
load_data.drop_all_collections(database)
stats = load_data.load_all_data(database)

### Pregled broja dokumenata po kolekciji

In [ ]:
print("Broj dokumenata po kolekciji:")
print("=" * 40)
for coll_name in ["requests", "locations", "vehicle_details",
                   "sanitation_details", "environment_details", "building_details"]:
    count = database[coll_name].count_documents({})
    print(f"  {coll_name}: {count:,}")

print(f"\nPrimer dokumenta iz 'requests':")
pprint(database["requests"].find_one())

print(f"\nPrimer dokumenta iz 'locations':")
pprint(database["locations"].find_one())

## 4. Kreiranje indeksa za inicijalnu šemu

In [ ]:
load_data.create_base_indexes(database)

### Pregled kreiranih indeksa

In [ ]:
for coll_name in ["requests", "locations", "vehicle_details",
                   "sanitation_details", "environment_details", "building_details"]:
    indexes = database[coll_name].index_information()
    print(f"\n{coll_name}:")
    for idx_name, idx_info in indexes.items():
        print(f"  {idx_name}: {idx_info['key']}")

## 5. Analiza upita — inicijalna (normalizovana) šema

---

### Q1: Infrastrukturna korelacija po ward-u

In [ ]:
# TODO: Korelacija rupa na putu i ugašenih svetala po ward-u. Top 10 sa kombinovanim problemima.

### Q2: Zanemarene community areas

In [ ]:
# TODO: Community areas gde je prosečno vreme rešavanja infrastrukturnih problema više od 1.5x gradskog proseka.

### Q3: Sezonski obrasci

In [ ]:
# TODO: Sezonski obrasci za rupe i grafite po ward-u. Mesec sa najviše prijava i sezonski indeks.

### Q4: Problematični blokovi

In [ ]:
# TODO: Adrese sa 5+ različitih tipova žalbi u jednoj godini.

### Q5: Efikasnost po policijskom distriktu

In [ ]:
# TODO: Efikasnost rešavanja po policijskom distriktu i kategoriji žalbe.

### Sačuvaj vremena izvršavanja base upita

In [ ]:
# TODO: base_times = {"Q1": time_q1, ...}

## 6. Optimizacija šeme

Migracija 6 normalizovanih kolekcija → 1 kolekcija sa embedded dokumentima.  
Koristi cache pristup: učita sve lookup podatke u memoriju, pa gradi nove dokumente.

In [ ]:
optimize_schema.migrate_to_embedded(database)

In [ ]:
optimize_schema.create_optimized_indexes(database)

In [ ]:
print(f"Broj dokumenata u optimizovanoj kolekciji: {database['requests_optimized'].count_documents({}):,}")
print(f"\nPrimer embedded dokumenta:")
pprint(database["requests_optimized"].find_one())

### Pregled indeksa optimizovane kolekcije

In [ ]:
indexes = database["requests_optimized"].index_information()
print("Indeksi na 'requests_optimized':")
for idx_name, idx_info in indexes.items():
    print(f"  {idx_name}: {idx_info['key']}")

## 7. Optimizovani upiti (embedded šema, bez $lookup)

---

### Q1 (optimizovano): Infrastrukturna korelacija po ward-u

In [ ]:
# TODO: optimizovana verzija Q1 (bez $lookup)

### Q2 (optimizovano): Zanemarene community areas

In [ ]:
# TODO: optimizovana verzija Q2 (bez $lookup)

### Q3 (optimizovano): Sezonski obrasci

In [ ]:
# TODO: optimizovana verzija Q3 (bez $lookup)

### Q4 (optimizovano): Problematični blokovi

In [ ]:
# TODO: optimizovana verzija Q4 (bez $lookup)

### Q5 (optimizovano): Efikasnost po policijskom distriktu

In [ ]:
# TODO: optimizovana verzija Q5 (bez $lookup)

## 8. Uporedna analiza performansi

In [ ]:
# TODO: poređenje base vs optimized vremena

## Zaključak

In [ ]:
# TODO